# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [2]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Augmente la mémoire pour éviter les crash sur les images
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")


Spark version : 4.1.1


## 1 - Chemins & constantes

In [3]:
TRAIN_PATH   = "./data/Train_5/"
TEST_PATH    = "./data/Test_5/"
OUTPUT_PREDS = "./output/predictions/" # seul Parquet écrit
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)


In [5]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [6]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])
decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .select(
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

c:\Users\Julien ANTOGNELLI\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Images train : 10
Images test  : 10
+----------+-------+--------------------+
|  image_id|  label|              pixels|
+----------+-------+--------------------+
|000139.jpg|tulipes|[180.0, 129.0, 79...|
|000137.jpg|tulipes|[184.0, 155.0, 50...|
|000063.jpg|    lys|[154.0, 2.0, 1.0,...|
|000140.jpg|tulipes|[118.0, 102.0, 84...|
|000061.jpg|    lys|[131.0, 119.0, 82...|
|000138.jpg|tulipes|[81.0, 15.0, 54.0...|
|000062.jpg|    lys|[205.0, 198.0, 16...|
|000064.jpg|    lys|[118.0, 145.0, 10...|
|000141.jpg|tulipes|[0.0, 72.0, 0.0, ...|
|000065.jpg|    lys|[164.0, 190.0, 21...|
+----------+-------+--------------------+

